In [2]:
import pandas as pd
import numpy as np
import os

# Robust path handling - works whether cwd is notebooks/ or project root
current_dir = os.getcwd()
if current_dir.endswith('notebooks'):
    data_path = '../data/raw/online_retail_II.xlsx'
else:
    data_path = 'data/raw/online_retail_II.xlsx'

print(f"Current working directory: {current_dir}")
print(f"Using data path: {data_path}")
print(f"File exists: {os.path.exists(data_path)}")

Current working directory: c:\Users\HARSHAVARDHAN\Desktop\retailpulse-analytics
Using data path: data/raw/online_retail_II.xlsx
File exists: True


In [3]:
import pandas as pd
import numpy as np
import os

# Robust path handling - works whether cwd is notebooks/ or project root
current_dir = os.getcwd()
if current_dir.endswith('notebooks'):
    data_path = '../data/raw/online_retail_II.xlsx'
else:
    data_path = 'data/raw/online_retail_II.xlsx'

print(f"Current working directory: {current_dir}")
print(f"Using data path: {data_path}")
print(f"File exists: {os.path.exists(data_path)}")

Current working directory: c:\Users\HARSHAVARDHAN\Desktop\retailpulse-analytics
Using data path: data/raw/online_retail_II.xlsx
File exists: True


In [4]:
# Load both sheets (2009-2010 and 2010-2011)
df_dict = pd.read_excel(data_path, sheet_name=None)
print("Sheet names:", df_dict.keys())

# Combine both years into one dataframe
df = pd.concat(df_dict.values(), ignore_index=True)
print(f"\nTotal rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nColumn names:\n{df.columns.tolist()}")

Sheet names: dict_keys(['Year 2009-2010', 'Year 2010-2011'])

Total rows: 1067371
Total columns: 8

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [5]:
# Basic info
print(df.info())
print("\n--- First 5 rows ---")
print(df.head())
print("\n--- Missing values per column ---")
print(df.isnull().sum())
print("\n--- Basic statistics ---")
print(df.describe())


<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB
None

--- First 5 rows ---
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS      

In [6]:
print("--- Missing values ---")
print(df.isnull().sum())
print("\n--- Missing % ---")
print((df.isnull().sum() / len(df) * 100).round(2))

print("\n--- Quantity range ---")
print(f"Min: {df['Quantity'].min()}, Max: {df['Quantity'].max()}")

print("\n--- Price range ---")
print(f"Min: {df['Price'].min()}, Max: {df['Price'].max()}")

print("\n--- Negative Quantity rows (likely returns) ---")
print((df['Quantity'] < 0).sum())

print("\n--- Zero or negative Price rows ---")
print((df['Price'] <= 0).sum())

print("\n--- Invoices starting with 'C' (cancellations) ---")
print(df['Invoice'].astype(str).str.startswith('C').sum())


--- Missing values ---
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

--- Missing % ---
Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.77
Country         0.00
dtype: float64

--- Quantity range ---
Min: -80995, Max: 80995

--- Price range ---
Min: -53594.36, Max: 38970.0

--- Negative Quantity rows (likely returns) ---
22950

--- Zero or negative Price rows ---
6207

--- Invoices starting with 'C' (cancellations) ---
19494


In [7]:
neg_qty = (df['Quantity'] < 0).sum()
zero_neg_price = (df['Price'] <= 0).sum()
price_min = df['Price'].min()
price_max = df['Price'].max()

print(f"Negative Quantity rows: {neg_qty}")
print(f"Zero/negative Price rows: {zero_neg_price}")
print(f"Price min: {price_min}, Price max: {price_max}")

Negative Quantity rows: 22950
Zero/negative Price rows: 6207
Price min: -53594.36, Price max: 38970.0


In [8]:
print(df[df['Price'] <= 0][['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']].head(15))
print("\nUnique descriptions for these rows:")
print(df[df['Price'] <= 0]['Description'].value_counts().head(15))


     Invoice StockCode         Description  Quantity  Price
263   489464     21733        85123a mixed       -96    0.0
283   489463     71477               short      -240    0.0
284   489467    85123A         21733 mixed      -192    0.0
470   489521     21646                 NaN       -50    0.0
3114  489655     20683                 NaN       -44    0.0
3161  489659     21350                 NaN       230    0.0
3162  489660     35956                lost     -1043    0.0
3168  489663    35605A             damages      -117    0.0
3731  489781     84292                 NaN        17    0.0
4296  489806     18010                 NaN      -770    0.0
4538  489820     21133     invcd as 84879?      -720    0.0
4566  489821    85049G                 NaN      -240    0.0
4674  489825     22076  6 RIBBONS EMPIRE          12    0.0
5904  489861       DOT      DOTCOM POSTAGE         1    0.0
6378  489882    35751C                 NaN        12    0.0

Unique descriptions for these rows:
Des

In [9]:
# ============================================
# DATA CLEANING PIPELINE
# ============================================

print(f"Starting rows: {len(df):,}")

# 1. Remove non-sale adjustment rows (Price <= 0)
df_clean = df[df['Price'] > 0].copy()
print(f"After removing Price <= 0: {len(df_clean):,}")

# 2. Separate cancellations (Invoice starts with 'C') into their own dataframe
#    Keep them for a "returns analysis" feature later, but exclude from main sales data
df_cancellations = df_clean[df_clean['Invoice'].astype(str).str.startswith('C')].copy()
df_sales = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')].copy()

print(f"Cancellation rows separated: {len(df_cancellations):,}")
print(f"Clean sales rows: {len(df_sales):,}")

# 3. Remove any remaining negative quantities in the sales set (edge cases)
negative_left = (df_sales['Quantity'] < 0).sum()
print(f"Remaining negative quantities in sales set: {negative_left}")
df_sales = df_sales[df_sales['Quantity'] > 0].copy()
print(f"Final clean sales rows: {len(df_sales):,}")

# 4. Add a total line value column (useful for RFM later)
df_sales['TotalPrice'] = df_sales['Quantity'] * df_sales['Price']

# 5. Check final missing values
print("\n--- Missing values in df_sales ---")
print(df_sales.isnull().sum())

Starting rows: 1,067,371
After removing Price <= 0: 1,061,164
Cancellation rows separated: 19,494
Clean sales rows: 1,041,670
Remaining negative quantities in sales set: 0
Final clean sales rows: 1,041,670

--- Missing values in df_sales ---
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    236121
Country             0
TotalPrice          0
dtype: int64


In [11]:
# ============================================
# SPLIT FOR DIFFERENT PURPOSES
# ============================================

# Dataset A: For demand forecasting (all sales, customer ID not required)
df_forecast = df_sales.copy()
print(f"Forecasting dataset: {len(df_forecast):,} rows")

# Dataset B: For customer segmentation/churn (must have Customer ID)
df_customers = df_sales.dropna(subset=['Customer ID']).copy()
df_customers['Customer ID'] = df_customers['Customer ID'].astype(int)
print(f"Customer-level dataset: {len(df_customers):,} rows")
print(f"Unique customers: {df_customers['Customer ID'].nunique():,}")

# Save both to processed folder
df_forecast.to_csv('data/processed/sales_for_forecasting.csv', index=False)
df_customers.to_csv('data/processed/sales_with_customers.csv', index=False)
df_cancellations.to_csv('data/processed/cancellations.csv', index=False)

print("\n✅ Saved 3 processed files to data/processed/")

Forecasting dataset: 1,041,670 rows
Customer-level dataset: 805,549 rows
Unique customers: 5,878

✅ Saved 3 processed files to data/processed/
